# Day 27 — Streaming replay: past-only scoring, hour by hour

Status: COMPLETE — `src/replay.py` walks two demo patients (septic p000011,
clean p003658), rebuilding each hour's window from past rows only and scoring
with the production LightGBM. 22/22 tests pass incl. future-truncation
invariance: truncating hours T+1.. leaves scores 1..T bit-identical.

In [1]:
import json
from pathlib import Path

demo = json.loads(Path("../models/replay_demo.json").read_text())
for p in demo:
    print(f"{p['pid']}: septic={p['septic']} onset@{p['clinical_onset_hour']} "
          f"first_alert@{p['first_alert_hour']} lead={p['lead_time_hours']}h "
          f"max_risk={p['max_risk']}")

p000011: septic=True onset@32 first_alert@2 lead=30h max_risk=0.7812
p003658: septic=False onset@None first_alert@1 lead=None max_risk=0.8559


In [2]:
sep, clean = demo
s = [p["risk"] for p in sep["trajectory"][:12]]
c = [p["risk"] for p in clean["trajectory"][:12]]
print("septic hourly risk (h1-12):", " ".join(f"{v:.2f}" for v in s))
print("max risk 0.78 lands at hour 31 — right at clinical onset (32), not 30h early.")
print("clean hourly risk (h1-12): ", " ".join(f"{v:.2f}" for v in c))
print("clean stay never drops quiet: 336h of 0.3-0.6 risk on a never-septic patient.")

septic hourly risk (h1-12): 0.40 0.17 0.30 0.31 0.22 0.36 0.30 0.53 0.40 0.28 0.16 0.46
max risk 0.78 lands at hour 31 — right at clinical onset (32), not 30h early.
clean hourly risk (h1-12):  0.80 0.34 0.32 0.39 0.58 0.44 0.61 0.44 0.56 0.47 0.45 0.39
clean stay never drops quiet: 336h of 0.3-0.6 risk on a never-septic patient.


## Honest read: the replay caught what batch metrics hid

1. **The '30h lead' is mostly hair-trigger.** First alert fires at hour 2 on a
   0.40 wobble; sustained high risk only builds near onset (peak 0.78 at h31).
   Lead time measured to *first crossing* flatters trigger-happy models — a
   deployment rule must require *sustained* elevation (e.g. 3 consecutive hours
   above threshold). Day 30 topic; noted, not implemented.
2. **Sparse early windows over-alert.** Hour 1 scores 0.80 on the clean patient:
   the model learned 'missing labs = sick' (Day 22's thesis) and a fresh
   admission with no labs yet looks maximally suspicious. The clean 336h stay
   then simmers at 0.3–0.6 forever — pure alert-fatigue fuel.
3. **This is why the simulator exists.** Batch ROC/PR on full frames showed
   neither behavior. Replay under deployment constraints is where streaming
   models actually get evaluated — everything before it was qualification.

## Handoff to Day 28

Wrap the scorer in FastAPI (`POST /score` on a trailing window, stateless) +
Dockerize. The replay's trajectory format becomes the API's contract test.